In [35]:
import pandas as pd
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 그래프 한글 깨짐 방지(Colab)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
from pathlib import Path

candidate_paths = [
    Path("data/fraudTrain.csv"),
    Path("../data/fraudTrain.csv"),
    Path("fraudTrain.csv.zip"),
    Path("fraudTrain.csv"),
]

train_path = next((p for p in candidate_paths if p.exists()), None)

if train_path is None:
    print("cwd:", Path.cwd())
    raise FileNotFoundError("fraudTrain.csv(.zip)를 찾지 못함")

df = pd.read_csv(train_path)


In [37]:
# 1. 결측치 확인
df.isnull().sum()

# 2. 중복 확인
df.duplicated().sum()

# 3. 자료형 확인
df.info()

# 4. 날짜형 변환
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

In [ ]:
df.head()

In [ ]:
df.info()  #데이터 타입 확인

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   Unnamed: 0             1296675 non-null  int64         
 1   trans_date_trans_time  1296675 non-null  datetime64[ns]
 2   cc_num                 1296675 non-null  int64         
 3   merchant               1296675 non-null  object        
 4   category               1296675 non-null  object        
 5   amt                    1296675 non-null  float64       
 6   first                  1296675 non-null  object        
 7   last                   1296675 non-null  object        
 8   gender                 1296675 non-null  object        
 9   street                 1296675 non-null  object        
 10  city                   1296675 non-null  object        
 11  state                  1296675 non-null  object        
 12  zip                    12966

In [ ]:
print(df.columns) #컬럼명 확인

In [ ]:
df.dtypes

In [38]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):

    R = 6371.0

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

In [ ]:
#!pip install haversine

In [39]:

offline_df = df[~df['category'].isin(['shopping_net', 'misc_net'])].copy()

print(offline_df.shape)
offline_df['category'].value_counts()

offline_df = offline_df.sort_values(
    ['cc_num', 'trans_date_trans_time']
)

offline_df['prev_lat'] = (
    offline_df.groupby('cc_num')['merch_lat']
    .shift(1)
)

offline_df['prev_long'] = (
    offline_df.groupby('cc_num')['merch_long']
    .shift(1)
)

offline_df['move_distance_km'] = offline_df.apply(
    lambda x: haversine_distance(
        x['prev_lat'],
        x['prev_long'],
        x['merch_lat'],
        x['merch_long']
    )
    if pd.notnull(x['prev_lat']) else np.nan,
    axis=1
)

offline_df['time_diff_sec'] = (
    offline_df.groupby('cc_num')['trans_date_trans_time']
    .diff()
    .dt.total_seconds()
)
offline_df['speed_kmh'] = np.where(
    offline_df['time_diff_sec'] > 0,
    offline_df['move_distance_km'] / (offline_df['time_diff_sec'] / 3600),
    np.nan
)


(1135845, 23)


In [9]:
offline_df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,trans_num,unix_time,merch_lat,merch_long,is_fraud,prev_lat,prev_long,move_distance_km,time_diff_sec,speed_kmh
2724,2724,2019-01-02 08:44:57,60416207185,fraud_Berge LLC,gas_transport,52.94,Mary,Diaz,F,9886 Anita Drive,...,498120fc45d277f7c88e3dba79c33865,1325493897,42.018766,-109.044172,0,NaN,NaN,NaN,NaN,NaN
2726,2726,2019-01-02 08:47:36,60416207185,fraud_Luettgen PLC,gas_transport,82.08,Mary,Diaz,F,9886 Anita Drive,...,95f514bb993151347c7acdf8505c3d62,1325494056,42.961335,-109.157564,0,42.018766,-109.044172,105.220439,159.0,2382.349553
2882,2882,2019-01-02 12:38:14,60416207185,fraud_Daugherty LLC,kids_pets,34.79,Mary,Diaz,F,9886 Anita Drive,...,4f0c1a14e0aa7eb56a490780ef9268c5,1325507894,42.228227,-108.747683,0,42.961335,-109.157564,88.152283,13838.0,22.933099
2907,2907,2019-01-02 13:10:46,60416207185,fraud_Beier and Sons,home,27.18,Mary,Diaz,F,9886 Anita Drive,...,3b2ebd3af508afba959640893e1e82bc,1325509846,43.321745,-108.091143,0,42.228227,-108.747683,132.876773,1952.0,245.059622
4337,4337,2019-01-03 17:05:10,60416207185,fraud_Conroy-Emard,food_dining,8.43,Mary,Diaz,F,9886 Anita Drive,...,da5ef053fa971418ab30fc72509c66f8,1325610310,42.871477,-109.160268,0,43.321745,-108.091143,100.209995,100464.0,3.590898


In [9]:
offline_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1135845 entries, 2724 to 1296427
Data columns (total 28 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   Unnamed: 0             1135845 non-null  int64         
 1   trans_date_trans_time  1135845 non-null  datetime64[ns]
 2   cc_num                 1135845 non-null  int64         
 3   merchant               1135845 non-null  object        
 4   category               1135845 non-null  object        
 5   amt                    1135845 non-null  float64       
 6   first                  1135845 non-null  object        
 7   last                   1135845 non-null  object        
 8   gender                 1135845 non-null  object        
 9   street                 1135845 non-null  object        
 10  city                   1135845 non-null  object        
 11  state                  1135845 non-null  object        
 12  zip                    1135845

In [40]:
import numpy as np
import pandas as pd

# 날짜형 변환
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

# 정렬
df = df.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)

# 결과 저장
df['count_30min'] = 0

# 카드별 계산
for cc_num, idx in df.groupby('cc_num').groups.items():

    times = df.loc[idx, 'trans_date_trans_time'].values.astype('datetime64[s]')

    left = np.searchsorted(
        times,
        times - np.timedelta64(30, 'm')
    )

    right = np.arange(len(times))

    df.loc[idx, 'count_30min'] = right - left + 1



In [41]:
#########추가
offline_df = offline_df[
    [
        'trans_num',
        'move_distance_km',
        'time_diff_sec',
        'speed_kmh'
    ]
]
df = df.merge(
    offline_df,
    on='trans_num',
    how='left'
)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 27 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   Unnamed: 0             1296675 non-null  int64         
 1   trans_date_trans_time  1296675 non-null  datetime64[ns]
 2   cc_num                 1296675 non-null  int64         
 3   merchant               1296675 non-null  object        
 4   category               1296675 non-null  object        
 5   amt                    1296675 non-null  float64       
 6   first                  1296675 non-null  object        
 7   last                   1296675 non-null  object        
 8   gender                 1296675 non-null  object        
 9   street                 1296675 non-null  object        
 10  city                   1296675 non-null  object        
 11  state                  1296675 non-null  object        
 12  zip                    12966

In [43]:
# 사용할 컬럼만 선택
final_df = df[
    [
        'lat',
        'long',
        'dob',
        'trans_date_trans_time',
        'cc_num',
        'merchant',
        'amt',
        'merch_lat',
        'merch_long',
        'category',
        'is_fraud',
        'trans_num',
        'count_30min'
    ]
].copy()

# 파생변수 추가 (_x 사용)
final_df['move_distance_km'] = df['move_distance_km']
final_df['time_diff_sec'] = df['time_diff_sec']
final_df['speed_kmh'] = df['speed_kmh']

# 컬럼 순서 확인
final_df = final_df[
    [
        'trans_num',
        'trans_date_trans_time',
        'cc_num',
        'merchant',
        'category',
        'amt',
        'lat',
        'long',
        'dob',
        'merch_lat',
        'merch_long',
        'move_distance_km',
        'time_diff_sec',
        'speed_kmh',
        'count_30min',
        'is_fraud'
    ]
]

# 확인
print(final_df.shape)
print(final_df.info())

(1296675, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 16 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   trans_num              1296675 non-null  object        
 1   trans_date_trans_time  1296675 non-null  datetime64[ns]
 2   cc_num                 1296675 non-null  int64         
 3   merchant               1296675 non-null  object        
 4   category               1296675 non-null  object        
 5   amt                    1296675 non-null  float64       
 6   lat                    1296675 non-null  float64       
 7   long                   1296675 non-null  float64       
 8   dob                    1296675 non-null  datetime64[ns]
 9   merch_lat              1296675 non-null  float64       
 10  merch_long             1296675 non-null  float64       
 11  move_distance_km       1134862 non-null  float64       
 12  time_diff_sec 

In [44]:
df = df.sort_values(
    ['cc_num', 'trans_date_trans_time']
).copy()

# ---------- 0. 사전 준비 (날짜 자료형 변환) ----------
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])



# ---------- 2. is_online (온라인 결제 여부) ----------
df['is_online'] = df['category'].str.endswith('_net')


# ---------- 3. time_diff_sec, is_high_amt ----------

df['is_high_amt'] = (df['amt'] >= 500).astype(int)


# ---------- 4. recent_24h_txn_count, recent_24h_high_amt_count ----------
df_time_idx = df.set_index('trans_date_trans_time')

df['recent_24h_txn_count'] = (
    df_time_idx.groupby('cc_num')['amt']
    .rolling('24h', closed='left', min_periods=0).count().values
)
df['recent_24h_high_amt_count'] = (
    df_time_idx.groupby('cc_num')['is_high_amt']
    .rolling('24h', closed='left', min_periods=0).sum().values
)


# ---------- 5. category_recent_fraud_rate (업종별 최근 7일 사기율) ----------

# ---------- 5. category_recent_fraud_rate ----------

# 먼저 정렬
df = df.sort_values(['category', 'trans_date_trans_time'])

# 그 다음 index 설정
df_time_idx = df.set_index('trans_date_trans_time')

df['category_recent_fraud_rate'] = (
    df_time_idx.groupby('category')['is_fraud']
    .rolling('7D', closed='left')
    .mean()
    .values
)

overall_fraud_rate = df['is_fraud'].mean()
df['category_recent_fraud_rate'] = (
    df['category_recent_fraud_rate']
    .fillna(overall_fraud_rate)
)
overall_fraud_rate = df['is_fraud'].mean()
df['category_recent_fraud_rate'] = df['category_recent_fraud_rate'].fillna(overall_fraud_rate)


# ---------- 6. distance_km (고객 평소 위치 vs 가맹점 위치 거리) ----------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distance_km_2'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
df.loc[df['is_online'], 'distance_km_2'] = np.nan



# ---------- 7. merchant_shift_km, time_diff_hr, speed_kmh (이동 속도) ----------
df_offline = df[~df['is_online']].sort_values(['cc_num', 'trans_date_trans_time']).copy()

df_offline['prev_merch_lat'] = df_offline.groupby('cc_num')['merch_lat'].shift(1)
df_offline['prev_merch_long'] = df_offline.groupby('cc_num')['merch_long'].shift(1)
df_offline['prev_trans_time'] = df_offline.groupby('cc_num')['trans_date_trans_time'].shift(1)

df_offline['merchant_shift_km'] = haversine(
    df_offline['prev_merch_lat'], df_offline['prev_merch_long'],
    df_offline['merch_lat'], df_offline['merch_long']
)
df_offline['time_diff_hr'] = (
    (df_offline['trans_date_trans_time'] - df_offline['prev_trans_time']).dt.total_seconds() / 3600
)
df_offline['speed_2'] = df_offline['merchant_shift_km'] / df_offline['time_diff_hr'].clip(lower=1 / 60)

# 필요한 컬럼만 추출
offline_features = df_offline[
    ['trans_num', 'merchant_shift_km', 'time_diff_hr', 'speed_2']
]

# 원본에 병합
df = df.merge(
    offline_features,
    on='trans_num',
    how='left'
)


# ————— 8. hour, dayofweek, year_month —————
df['hour'] = df['trans_date_trans_time'].dt.hour
df['dayofweek'] = df['trans_date_trans_time'].dt.day_name()
df['year_month'] = df['trans_date_trans_time'].dt.to_period('M').astype(str)

##모델링용이라면 숫자가 더 낫다
df['dayofweek'] = df['trans_date_trans_time'].dt.dayofweek

In [45]:
# ===============================
# 최종 확인
# ===============================

print("=" * 50)
print("데이터 크기")
print(df.shape)

print("\n" + "=" * 50)
print("컬럼 목록")
print(df.columns.tolist())

print("\n" + "=" * 50)
print("자료형")
print(df.dtypes)

print("\n" + "=" * 50)
print("결측치 개수")
print(df.isnull().sum())

print("\n" + "=" * 50)
print("trans_num 중복 개수")
print(df['trans_num'].duplicated().sum())

print("\n" + "=" * 50)
print("온라인 거래 개수")
print(df['is_online'].sum())

print("\n" + "=" * 50)
print("온라인 거래의 speed_2 결측 여부")
print(df.loc[df['is_online'], 'speed_2'].isna().all())

print("\n" + "=" * 50)
print("온라인 거래의 merchant_shift_km 결측 여부")
print(df.loc[df['is_online'], 'merchant_shift_km'].isna().all())

print("\n" + "=" * 50)
print("온라인 거래의 distance_km_2 결측 여부")
print(df.loc[df['is_online'], 'distance_km_2'].isna().all())

print("\n" + "=" * 50)
print(df.head())

데이터 크기
(1296675, 39)

컬럼 목록
['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'count_30min', 'move_distance_km', 'time_diff_sec', 'speed_kmh', 'is_online', 'is_high_amt', 'recent_24h_txn_count', 'recent_24h_high_amt_count', 'category_recent_fraud_rate', 'distance_km_2', 'merchant_shift_km', 'time_diff_hr', 'speed_2', 'hour', 'dayofweek', 'year_month']

자료형
Unnamed: 0                             int64
trans_date_trans_time         datetime64[ns]
cc_num                                 int64
merchant                              object
category                              object
amt                                  float64
first                                 object
last                                  object
gender                                object
street                                

In [47]:


# _x → 원래 이름으로 변경
df = df.rename(columns={
    'move_distance_km_x': 'move_distance_km',
    'time_diff_sec_x': 'time_diff_sec',
    'speed_kmh_x': 'speed_kmh'
})

In [48]:
df = df.drop(columns=[
    'is_online',
    'is_high_amt'
])

In [49]:
print(df.columns.tolist())

['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'count_30min', 'move_distance_km', 'time_diff_sec', 'speed_kmh', 'recent_24h_txn_count', 'recent_24h_high_amt_count', 'category_recent_fraud_rate', 'distance_km_2', 'merchant_shift_km', 'time_diff_hr', 'speed_2', 'hour', 'dayofweek', 'year_month']


In [50]:

# 카드별 평균 거래금액
df["customer_mean_amt"] = (
    df.groupby("cc_num")["amt"]
      .transform("mean")
)

# 거래 연도
df["trans_year"] = df["trans_date_trans_time"].dt.year
# 거래 월
df["trans_month"] = df["trans_date_trans_time"].dt.month

# 심야 거래 여부 (00시~05시)
df["night_transaction"] = (
    (df["hour"] <= 5)
).astype(int)


# 고객 거래금액 표준편차
df["customer_std_amt"] = (
    df.groupby("cc_num")["amt"]
      .transform("std")
      .fillna(0)
)

# 현재 거래금액이 평균의 몇 배인지
df["amt_ratio_to_mean"] = (
    df["amt"] /
    df["customer_mean_amt"].replace(0,np.nan)
)

df["amt_ratio_to_mean"] = (
    df["amt_ratio_to_mean"]
    .replace([np.inf,-np.inf],0)
    .fillna(0)
)


# 고객별 거래금액 Z-score
df["amt_zscore_card"] = (
    (df["amt"] - df["customer_mean_amt"]) /
    df["customer_std_amt"]
)

df["amt_zscore_card"] = (
    df["amt_zscore_card"]
    .replace([np.inf,-np.inf],0)
    .fillna(0)
)
# 고객별 전체 거래 횟수
df["customer_transaction_count"] = (
    df.groupby("cc_num")["cc_num"]
      .transform("count")
)
# 고객-업종별 이용 횟수
df["customer_category_count"] = (
    df.groupby(["cc_num","category"])["category"]
      .transform("count")
)


# 월별 총 거래 건수
monthly_transaction = (
    df.groupby("trans_month")
      .size()
      .reset_index(name="transaction_count")
)

monthly_transaction
monthly_fraud = (
    df[df["is_fraud"] == 1]
      .groupby("trans_month")
      .size()
      .reset_index(name="fraud_count")
)

fraud_zscore = (
    df[df["is_fraud"] == 1]
      .groupby("cc_num")
      .agg(
          fraud_count=("is_fraud", "sum"),
          mean_amt=("customer_mean_amt", "first"),
          std_amt=("customer_std_amt", "first"),
          mean_zscore=("amt_zscore_card", "mean"),
          max_zscore=("amt_zscore_card", "max")
      )
      .sort_values("mean_zscore", ascending=False)
)








In [51]:
print(df[['move_distance_km', 'merchant_shift_km']].corr())

print(df[['speed_kmh', 'speed_2']].corr())

                   move_distance_km  merchant_shift_km
move_distance_km           1.000000           0.965826
merchant_shift_km          0.965826           1.000000
           speed_kmh   speed_2
speed_kmh   1.000000  0.406349
speed_2     0.406349  1.000000


In [52]:
drop_cols = [
    "Unnamed: 0",
    "first",
    "last",
    "gender",
    "street",
    "city",
    "state",
    "zip",
    "city_pop",
    "job",
    "unix_time"
]

df = df.drop(columns=drop_cols)

In [53]:
df.to_csv(
    'fraudTrain_feature_engineering.zip',
    index=False,
    compression='zip'
)

from google.colab import files

files.download('fraudTrain_feature_engineering.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>